In [1]:
import json
from dotenv import load_dotenv
import gzip
import os
import re
import requests
import wget
from tqdm import tqdm

from evaluation.config import PROJECT_DIR
from granuscore.loader import JSONLineReader

load_dotenv()

True

In [13]:
# modify these
API_KEY = os.getenv('S2_API_KEY')
DATASET_NAME = "s2orc"
LOCAL_PATH = f"{PROJECT_DIR}/data/s2orc/"
os.makedirs(LOCAL_PATH, exist_ok=True)

# get latest release's ID
response = requests.get("https://api.semanticscholar.org/datasets/v1/release/latest").json()
RELEASE_ID = response["release_id"]
print(f"Latest release ID: {RELEASE_ID}")

# get the download links for the s2orc dataset; needs to pass API key through `x-api-key` header
# download via wget. this can take a while...
response = requests.get(f"https://api.semanticscholar.org/datasets/v1/release/{RELEASE_ID}/dataset/{DATASET_NAME}/", headers={"x-api-key": API_KEY}).json()
for url in tqdm(response["files"][3:4]):
    match = re.match(r"https://ai2-s2ag.s3.amazonaws.com/staging/(.*)/s2orc/(.*).gz(.*)", url)
    assert match.group(1) == RELEASE_ID
    SHARD_ID = match.group(2)
    wget.download(url, out=os.path.join(LOCAL_PATH, f"{SHARD_ID}.gz"))
print("Downloaded all shards.")

Latest release ID: 2025-11-25


100%|██████████| 1/1 [02:18<00:00, 138.47s/it]

Downloaded all shards.


In [14]:
def extract_section(text, section_name):
    pattern = rf"{section_name}\n\n(.*?)(?:\n{{3,}}|$)"
    match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else None

papers = []
for filename in os.listdir(f'{PROJECT_DIR}/eval/data/s2orc/'):
    if not filename.endswith(".gz"):
        continue
    with gzip.open(f"{PROJECT_DIR}/eval/data/s2orc/{filename}", "rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            paper = json.loads(line)
            external_ids = paper.get("externalids") or {}
            doi = external_ids.get("doi")
            if not doi:
                continue

            text = paper['content']['text']
            if not text:
                continue

            if "Introduction\n\n" in text and "Related Work\n\n" in text and "Conclusion\n\n" in text:
                intro = extract_section(text, "Introduction")
                related = extract_section(text, "Related Work")
                conclusion = extract_section(text, "Conclusion")
                papers.append({'doi': doi, 'introduction': intro, 'related_work': related, 'conclusion': conclusion})

In [15]:
OUTPUT_FILE = f'{PROJECT_DIR}/data/s2orc/papers.jsonl'
JSONLineReader().write(OUTPUT_FILE, papers[:1000])

### Filter papers

In [2]:
from granuscore.claim_splitter import SpacyNounPhraseSplitter

FILTERED_OUTPUT_FILE = f'{PROJECT_DIR}/data/s2orc/filtered_papers-final.jsonl'
papers = JSONLineReader().read(f'{PROJECT_DIR}/data/s2orc/papers.jsonl')
splitter = SpacyNounPhraseSplitter(skip_figures=True, skip_brackets=True, skip_latex=True, skip_arxiv_meta=True, skip_pdf_noise=True, skip_url_pattern=True)

def select_paragraph(
    text: str,
    min_facts: int = 10,
    skip_first: bool = False,
):
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    # 1) try paragraphs starting from index 1
    start_idx = 1 if skip_first else 0
    for p in paragraphs[start_idx:]:
        facts = splitter.get_referential_units(p)[0]
        if len(facts['referential_units']) >= min_facts:
            return facts, p

    # 2) fallback: paragraph 0
    if skip_first and paragraphs:
        facts = splitter.get_referential_units(paragraphs[0])[0]
        if len(facts['referential_units']) >= min_facts:
            return facts, paragraphs[0]

    return None, None

for paper in papers:
    intro_facts, intro_par = select_paragraph(paper.get('introduction', ''))
    rel_facts, rel_par = select_paragraph(paper.get('related_work', ''), skip_first=True)

    if intro_par is None or rel_par is None:
       print('Skipping', paper['doi'])
       continue

    JSONLineReader().write(FILTERED_OUTPUT_FILE, [{'doi': paper['doi'], 'introduction': paper.get('introduction'), 'introduction_par': intro_par, 'related_work_par': rel_par, 'related_work': paper.get('related_work')}])

Skipping 10.18653/v1/2024.argmining-1.6
Skipping 10.48550/arxiv.2312.04008
Skipping 10.48550/arxiv.2506.14825
Skipping 10.1155/2022/1357182
Skipping 10.1609/aaai.v39i23.34625
Skipping 10.48550/arxiv.2507.08802
Skipping 10.48550/arxiv.2208.14743
Skipping 10.48550/arxiv.2506.06742
Skipping 10.1007/978-3-030-29387-1_40
Skipping 10.1109/fg47880.2020.00085
Skipping 10.1109/cvpr52688.2022.01506
Skipping 10.1109/cvpr42600.2020.01041
Skipping 10.1007/978-3-030-49418-6_11
Skipping 10.48550/arxiv.2408.02509
Skipping 10.1109/enic.2016.017
Skipping 10.48550/arxiv.2508.03458
Skipping 10.1109/fg47880.2020.00085
Skipping 10.1145/3674883
Skipping 10.48550/arxiv.2509.25465
Skipping 10.1109/iccv51070.2023.00353
Skipping 10.13052/jcsm2245-1439.1026
Skipping 10.1145/3706598.3714280
